---
## 📝 Tarea — Clasificación supervisada de un glaciar peruano con SVM

Aplica el **mismo flujo de trabajo** de la sección 7, pero ahora sobre un **glaciar peruano**, usando SVM (en lugar de K-Means no supervisado de la tarea anterior).

**Glaciar sugerido:** Coropuna (Arequipa) — el mismo que usaste en la tarea de K-Means, para poder comparar ambos enfoques sobre la misma área. Puedes usar otro glaciar si lo prefieres (Pastoruri en Áncash, Quelccaya en Cusco, algún nevado de la Cordillera Vilcanota, etc.) — solo ajusta las coordenadas.

### Clases sugeridas para el glaciar

| Código | Clase |
|--------|-------|
| 0 | Agua (lagunas proglaciares) |
| 1 | Nieve / Hielo |
| 2 | Roca expuesta / morrena |
| 3 | Pastizal ralo (zonas bajas) |

### Pasos a seguir

1. **Definir el área de estudio** del glaciar (celda de partida más abajo, reutiliza las coordenadas de tu tarea de K-Means).
2. **Cargar y visualizar** un composito Sentinel-2 de época seca.
3. **Recolectar puntos de entrenamiento** (clic interactivo o predefinidos) para las 4 clases — recuerda **verificar visualmente** cada punto antes de entrenar.
4. **Extraer valores espectrales y normalizar** (igual que hicimos para la Laguna Junín).
5. **Entrenar al menos dos clasificadores SVM**: uno lineal y uno RBF (con y sin normalizar, para comparar igual que en la sección 7).
6. **Clasificar la imagen completa** y visualizar en el mapa con leyenda.
7. **Comparar con tu resultado de K-Means** (tarea anterior): ¿coinciden las clases? ¿Cuál te parece más confiable y por qué?

> **Pista clave:** la nieve/hielo es muy brillante en el visible y muy oscura en SWIR — un contraste fuerte que debería facilitar la separación con SVM lineal incluso sin kernel RBF.

In [1]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project="cda-ambiental")  # Usa tu proyecto habilitado en EE

# Glaciar Coropuna (Arequipa): oeste, sur, este, norte
LON_MIN_G, LAT_MIN_G = -72.766883, -15.650000
LON_MAX_G, LAT_MAX_G = -72.508209, -15.417441

area_glaciar = ee.Geometry.Rectangle(
    [LON_MIN_G, LAT_MIN_G, LON_MAX_G, LAT_MAX_G],
    geodesic=False
)

def enmascarar_nubes_hls(imagen):
    qa = imagen.select("Fmask")

    # Excluir nubes, píxeles adyacentes y sombras (bits 1, 2 y 3).
    # Conservar nieve/hielo (bit 4) para estudiar el glaciar.
    mascara = qa.bitwiseAnd(
        (1 << 1) | (1 << 2) | (1 << 3)
    ).eq(0)

    return imagen.updateMask(mascara).select(
        ["B2", "B3", "B4", "B5", "B6", "B7"],
        ["Azul", "Verde", "Rojo", "NIR", "SWIR1", "SWIR2"]
    )

coleccion_g = (
    ee.ImageCollection("NASA/HLS/HLSL30/v002")
    .filterBounds(area_glaciar)
    # La fecha final es exclusiva: incluye todo septiembre.
    .filterDate("2025-05-01", "2025-10-01")
)

n_imagenes = coleccion_g.size().getInfo()
print(f"Imágenes disponibles: {n_imagenes}")

if n_imagenes == 0:
    raise ValueError("No hay imágenes para el área y las fechas seleccionadas.")

composito_g = (
    coleccion_g
    .map(enmascarar_nubes_hls)
    .median()
    # No aplicar .multiply(0.0001) a esta colección en Earth Engine.
    .clip(area_glaciar)
)

Map_g = geemap.Map(basemap="Esri.WorldTopoMap")
Map_g.centerObject(area_glaciar, zoom=11)
Map_g.addLayer(
    composito_g,
    {
        "bands": ["Rojo", "Verde", "Azul"],
        "min": 0,
        "max": 0.6,
        "gamma": 1.2
    },
    "HLS Landsat 8/9 — Glaciar Coropuna"
)
Map_g

c:\Users\luisc\miniconda3\envs\cda\lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.22) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Imágenes disponibles: 35


Map(center=[-15.533711192277233, -72.63754599999888], controls=(WidgetControl(options=['position', 'transparen…

### Tu turno

Completa las celdas siguientes siguiendo el mismo patrón usado para la Laguna Junín (sección 7).

In [2]:
# ── Paso 3: recolección interactiva de puntos de entrenamiento (glaciar) ──
import ipywidgets as widgets
from IPython.display import display
from ipyleaflet import Marker, AwesomeIcon

# ── Definiciones de clases del glaciar ──
clases_glaciar = {
    'Agua': 0,
    'Nieve/Hielo': 1,
    'Roca/Morrena': 2,
    'Pastizal ralo': 3,
}
nombres_glaciar = {v: k for k, v in clases_glaciar.items()}
colores_marcador = {0: 'blue', 1: 'white', 2: 'black', 3: 'green'}

# Almacenamiento de puntos: {código_clase: [(lon, lat), ...]}
puntos_glaciar = {v: [] for v in clases_glaciar.values()}

# ── Menú desplegable y registro de salida ──
dropdown_glaciar = widgets.Dropdown(
    options=list(clases_glaciar.keys()),
    value='Agua',
    description='Clase:',
    style={'description_width': 'initial'}
)
log_glaciar = widgets.Output()

# ── Controlador de clics ──
def clic_glaciar(**kwargs):
    if kwargs.get('type') != 'click':
        return
    lat, lon = kwargs['coordinates']
    codigo = clases_glaciar[dropdown_glaciar.value]
    puntos_glaciar[codigo].append((lon, lat))

    icono = AwesomeIcon(name='map-marker', marker_color=colores_marcador[codigo])
    Map_g.add(Marker(location=(lat, lon), icon=icono, draggable=False))

    with log_glaciar:
        total = sum(len(v) for v in puntos_glaciar.values())
        print(f"[{dropdown_glaciar.value}] Punto añadido ({lon:.4f}, {lat:.4f}) | Total: {total} puntos")

Map_g.on_interaction(clic_glaciar)

# ── Botón para borrar ──
boton_borrar_g = widgets.Button(description='Borrar todos los puntos', button_style='warning')

def borrar_glaciar(b):
    for k in puntos_glaciar:
        puntos_glaciar[k] = []
    with log_glaciar:
        log_glaciar.clear_output()
        print("Todos los puntos borrados. (Los marcadores permanecen en el mapa.)")

boton_borrar_g.on_click(borrar_glaciar)

print("Haz clic en el mapa del glaciar para obtener puntos de entrenamiento.")
print("Selecciona una clase del menú desplegable y luego haz clic en el mapa.\n")
display(widgets.HBox([dropdown_glaciar, boton_borrar_g]))
display(log_glaciar)

Haz clic en el mapa del glaciar para obtener puntos de entrenamiento.
Selecciona una clase del menú desplegable y luego haz clic en el mapa.



Output()

In [30]:
# ── Paso 4: construir ee.FeatureCollection y extraer valores espectrales (+ z-score) ──
# Verificación: se requieren al menos 3 puntos por clase antes de entrenar
faltan = [nombres_glaciar[c] for c, pts in puntos_glaciar.items() if len(pts) < 3]
if faltan:
    raise ValueError(f"Recolecta al menos 3 puntos para cada clase. Faltan: {faltan}")

puntos_g = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([lon, lat]), {'landcover': codigo})
    for codigo, pts in puntos_glaciar.items()
    for lon, lat in pts
])
print(f"Total puntos de entrenamiento (glaciar): {puntos_g.size().getInfo()}")

bandas_g = ['Azul', 'Verde', 'Rojo', 'NIR', 'SWIR1', 'SWIR2']
bandas_g_norm = [b + '_norm' for b in bandas_g]

# Valores espectrales crudos en los puntos (HLS L30 = 30 m)
datos_g = composito_g.select(bandas_g).sampleRegions(
    collection=puntos_g,
    properties=['landcover'],
    scale=30
).filter(ee.Filter.notNull(bandas_g))

# Media y desviación estándar sobre el área del glaciar
medias_g = composito_g.select(bandas_g).reduceRegion(
    reducer=ee.Reducer.mean(), geometry=area_glaciar, scale=30, maxPixels=1e9
)
desv_g = composito_g.select(bandas_g).reduceRegion(
    reducer=ee.Reducer.stdDev(), geometry=area_glaciar, scale=30, maxPixels=1e9
)

# Normalización z-score: (x - media) / desviación estándar
composito_g_norm = ee.Image.cat([
    composito_g.select(b).subtract(ee.Number(medias_g.get(b))).divide(ee.Number(desv_g.get(b))).rename(b + '_norm')
    for b in bandas_g
])

datos_g_norm = composito_g_norm.sampleRegions(
    collection=puntos_g,
    properties=['landcover'],
    scale=30
).filter(ee.Filter.notNull(bandas_g_norm))

print("Datos de entrenamiento extraídos y normalizados (z-score).")

Total puntos de entrenamiento (glaciar): 100
Datos de entrenamiento extraídos y normalizados (z-score).


In [31]:
# ── Paso 5: entrenar SVM lineal y RBF (con y sin normalizar) ──
# 1) Núcleo lineal (bandas crudas)
svm_g_lineal = ee.Classifier.libsvm(
    kernelType='LINEAR', cost=10
).train(datos_g, classProperty='landcover', inputProperties=bandas_g)

# 2) Núcleo RBF sin normalizar (mismo gamma que en la Laguna Junín)
svm_g_rbf = ee.Classifier.libsvm(
    kernelType='RBF', cost=10, gamma=0.5
).train(datos_g, classProperty='landcover', inputProperties=bandas_g)

# 3) Núcleo RBF normalizado (z-score + gamma bajo)
svm_g_rbf_norm = ee.Classifier.libsvm(
    kernelType='RBF', cost=10, gamma=0.01
).train(datos_g_norm, classProperty='landcover', inputProperties=bandas_g_norm)

print("Tres clasificadores SVM entrenados para el glaciar:")
print(" 1. Lineal          — bandas crudas")
print(" 2. RBF sin escalar — bandas crudas, gamma=0.5")
print(" 3. RBF normalizado — z-score,       gamma=0.01")

Tres clasificadores SVM entrenados para el glaciar:
 1. Lineal          — bandas crudas
 2. RBF sin escalar — bandas crudas, gamma=0.5
 3. RBF normalizado — z-score,       gamma=0.01


In [32]:
# ── Paso 6: clasificar la imagen completa y visualizar con leyenda ──
clasi_g_lineal   = composito_g.select(bandas_g).classify(svm_g_lineal)
clasi_g_rbf      = composito_g.select(bandas_g).classify(svm_g_rbf)        # bandas crudas
clasi_g_rbf_norm = composito_g_norm.classify(svm_g_rbf_norm)              # bandas normalizadas

# Agua, Nieve/Hielo, Roca/Morrena, Pastizal ralo
paleta_g = ['#1f78b4', '#ffffff', '#5c5c5c', '#7ec841']
nombres_leyenda_g = ['Agua', 'Nieve/Hielo', 'Roca/Morrena', 'Pastizal ralo']

Map_g2 = geemap.Map(basemap="Esri.WorldTopoMap")
Map_g2.centerObject(area_glaciar, zoom=13)
Map_g2.addLayer(composito_g,
                {'bands': ['Rojo', 'Verde', 'Azul'], 'min': 0, 'max': 0.6},
                'Base — Glaciar')
Map_g2.addLayer(clasi_g_lineal,
                {'min': 0, 'max': 3, 'palette': paleta_g},
                'SVM Lineal')
Map_g2.addLayer(clasi_g_rbf,
                {'min': 0, 'max': 3, 'palette': paleta_g},
                'SVM RBF (sin normalizar)')
Map_g2.addLayer(clasi_g_rbf_norm,
                {'min': 0, 'max': 3, 'palette': paleta_g},
                'SVM RBF (normalizado)')
Map_g2.add_legend(
    title='Cobertura glaciar',
    keys=nombres_leyenda_g,
    colors=paleta_g
)
Map_g2

Map(center=[-15.533711192277233, -72.63754599999888], controls=(WidgetControl(options=['position', 'transparen…

In [36]:

CRS_COMP = "EPSG:32718"
ESCALA_COMP = 30
N_CLUSTERS = 4

muestra_km_g = composito_g.select(bandas_g).sample(
    region=area_glaciar,
    scale=ESCALA_COMP,
    projection=CRS_COMP,
    numPixels=5000,
    seed=42,
    geometries=False,
    tileScale=4
)

modelo_km_g = ee.Clusterer.wekaKMeans(
    nClusters=N_CLUSTERS,
    seed=42
).train(
    features=muestra_km_g,
    inputProperties=bandas_g
)

clasi_g_kmeans = (
    composito_g.select(bandas_g)
    .cluster(modelo_km_g)
    .rename("cluster")
    .clip(area_glaciar)
)

# Los colores representan grupos, todavía NO coberturas identificadas.
colores_km_g = ["#e41a1c", "#377eb8", "#ff7f00", "#984ea3"]

Map_km_g = geemap.Map(basemap="Esri.WorldTopoMap")
Map_km_g.centerObject(area_glaciar, 11)

Map_km_g.addLayer(
    composito_g,
    {"bands": ["Rojo", "Verde", "Azul"], "min": 0, "max": 0.6},
    "Imagen base"
)
Map_km_g.addLayer(
    clasi_g_kmeans,
    {"min": 0, "max": N_CLUSTERS - 1, "palette": colores_km_g},
    "K-Means: identificar nieve/hielo"
)
Map_km_g.add_legend(
    title="Grupos K-Means",
    keys=[f"Cluster {i}" for i in range(N_CLUSTERS)],
    colors=colores_km_g
)

Map_km_g

Map(center=[-15.533711192277226, -72.63754600000028], controls=(WidgetControl(options=['position', 'transparen…

In [ ]:
CLUSTERS_HIELO = [1]  

if not CLUSTERS_HIELO:
    raise ValueError(
        "Completa CLUSTERS_HIELO con el grupo o grupos de nieve/hielo."
    )

if not set(CLUSTERS_HIELO).issubset(range(N_CLUSTERS)):
    raise ValueError("Los números de cluster deben estar entre 0 y 3.")

# Máscara binaria K-Means: 1 = nieve/hielo, 0 = otra cobertura.
hielo_km_g = clasi_g_kmeans.remap(
    CLUSTERS_HIELO,
    [1] * len(CLUSTERS_HIELO),
    0
).rename("hielo")

# Variables reales de tu notebook.
clasificaciones_g = {
    "SVM lineal": clasi_g_lineal,
    "SVM RBF sin normalizar": clasi_g_rbf,
    "SVM RBF normalizado": clasi_g_rbf_norm
}

Map_acuerdo_g = geemap.Map(basemap="Esri.WorldTopoMap")
Map_acuerdo_g.centerObject(area_glaciar, 11)
Map_acuerdo_g.addLayer(
    composito_g,
    {"bands": ["Rojo", "Verde", "Azul"], "min": 0, "max": 0.6},
    "Imagen base"
)

resultados_g = []

for nombre, clasificacion in clasificaciones_g.items():
    hielo_svm = clasificacion.eq(1).rename("hielo")

    # Comparar únicamente donde ambos métodos tienen datos.
    mascara_comun = hielo_svm.mask().And(hielo_km_g.mask())
    s = hielo_svm.updateMask(mascara_comun)
    k = hielo_km_g.updateMask(mascara_comun)

    ambos = s.And(k)
    solo_svm = s.And(k.Not())
    solo_km = k.And(s.Not())

    areas_img = ee.Image.cat(
        s.rename("svm"),
        k.rename("kmeans"),
        ambos.rename("ambos"),
        solo_svm.rename("solo_svm"),
        solo_km.rename("solo_kmeans")
    ).multiply(ee.Image.pixelArea()).divide(1e6)

    areas = areas_img.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=area_glaciar,
        scale=ESCALA_COMP,
        crs=CRS_COMP,
        maxPixels=1e9,
        tileScale=4
    ).getInfo()

    if any(v is None for v in areas.values()):
        raise ValueError(f"No hay datos suficientes para comparar {nombre}.")

    union = areas["ambos"] + areas["solo_svm"] + areas["solo_kmeans"]
    iou = 100 * areas["ambos"] / union if union > 0 else float("nan")

    resultados_g.append({
        "Modelo": nombre,
        "Hielo SVM (km²)": areas["svm"],
        "Hielo K-Means (km²)": areas["kmeans"],
        "Coinciden (km²)": areas["ambos"],
        "Solo SVM (km²)": areas["solo_svm"],
        "Solo K-Means (km²)": areas["solo_kmeans"],
        "IoU (%)": iou
    })

    # 0 = ninguno; 1 = solo SVM; 2 = solo K-Means; 3 = ambos.
    acuerdo = s.add(k.multiply(2)).rename("acuerdo")
    Map_acuerdo_g.addLayer(
        acuerdo.updateMask(acuerdo.gt(0)),
        {
            "min": 1,
            "max": 3,
            "palette": ["#ff00ff", "#ffff00", "#00cc66"]
        },
        f"{nombre} vs K-Means",
        nombre == "SVM lineal"
    )

df_comparacion_g = pd.DataFrame(resultados_g)
display(df_comparacion_g.round(2))

Map_acuerdo_g.add_legend(
    title="Nieve/hielo: comparación",
    keys=["Solo SVM", "Solo K-Means", "Coinciden"],
    colors=["#ff00ff", "#ffff00", "#00cc66"]
)

# Texto con los resultados del SVM lineal.
fila = df_comparacion_g.iloc[0]
print(
    f"El SVM lineal identificó {fila['Hielo SVM (km²)']:.2f} km² "
    f"de nieve/hielo y K-Means {fila['Hielo K-Means (km²)']:.2f} km². "
    f"Ambos coincidieron en {fila['Coinciden (km²)']:.2f} km², "
    f"con una intersección sobre unión (IoU) de {fila['IoU (%)']:.1f} %. "
    "Estos valores corresponden al área con datos válidos en ambos mapas."
)

Map_acuerdo_g

,Modelo,Hielo SVM (km²),Hielo K-Means (km²),Coinciden (km²),Solo SVM (km²),Solo K-Means (km²),IoU (%)
0,SVM lineal,70.16,39.76,39.76,30.40,0,56.68
1,SVM RBF sin normalizar,70.16,39.76,39.76,30.40,0,56.68
2,SVM RBF normalizado,71.44,39.76,39.76,31.68,0,55.66


El SVM lineal identificó 70.16 km² de nieve/hielo y K-Means 39.76 km². Ambos coincidieron en 39.76 km², con una intersección sobre unión (IoU) de 56.7 %. Estos valores corresponden al área con datos válidos en ambos mapas.


Map(center=[-15.533711192277233, -72.63754599999888], controls=(WidgetControl(options=['position', 'transparen…

### Para tu informe

Responde brevemente (puedes usar esta misma celda en formato Markdown):

**1. ¿Qué kernel (lineal o RBF) dio un mapa más coherente para el glaciar? ¿Por qué crees que ocurrió eso?**
El SVM lineal y el RBF sin normalizar muestran resultados visualmente similares, con una zona central de nieve/hielo relativamente continua. El RBF normalizado presenta más píxeles clasificados como agua dentro y alrededor del glaciar, que podrían corresponder a confusiones con hielo oscuro o sombras. Por ello, el lineal y el RBF sin normalizar parecen más coherentes visualmente, aunque hace falta validación para determinar cuál es más preciso. La normalización modifica las distancias entre las muestras y, por tanto, puede cambiar el resultado del RBF si se mantienen sus parámetros.

**2. Compara el mapa de SVM con el de K-Means (tarea anterior, mismo glaciar). ¿Las clases de "nieve/hielo" coinciden espacialmente?**
Las clases de nieve/hielo coinciden parcialmente. El SVM lineal y el RBF sin normalizar identificaron 70,16 km², mientras que el grupo de K-Means seleccionado como nieve/hielo identificó 39,76 km². Según los valores reportados, toda esa superficie de K-Means está contenida en la clasificación SVM, pero SVM identifica 30,40 km² adicionales. La coincidencia espacial, medida mediante IoU, fue de 56,68 %. Para el RBF normalizado, el área de nieve/hielo fue de 71,44 km² y la IoU de 55,66 %. Por tanto, ambos métodos coinciden en una parte del glaciar, pero difieren en su extensión. Esto puede ocurrir porque K-Means separa superficies de nieve/hielo con distintas características espectrales en varios grupos, mientras que SVM puede reunirlas en una misma clase mediante las muestras etiquetadas. Estos resultados miden acuerdo entre métodos; no permiten determinar por sí solos cuál es más exacto.

**3. ¿Qué ventaja tiene SVM (supervisado) sobre K-Means (no supervisado) para este problema? ¿Y qué desventaja?**
La ventaja de SVM es que aprende de muestras etiquetadas de nieve/hielo, roca, agua y pastizal, orientando la clasificación hacia las coberturas de interés. K-Means forma grupos espectrales que después hay que interpretar. La desventaja de SVM es que requiere muestras representativas y un ajuste de parámetros: etiquetas incorrectas o muestras insuficientes pueden generar confusiones.